# Homework 3: Classificação (lead scoring)
ML Zoomcamp 2026

[Versão original no Google Colab](https://colab.research.google.com/drive/1mV63ZzE9gj5ZrqY5JzAU1N-C_1BJYjDP?usp=sharing)

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('course_lead_scoring_2026.csv')
print(df.isnull().sum())

categorical = list(df.dtypes[df.dtypes == 'object'].index)
numerical = [c for c in df.columns if c not in categorical and c != 'converted']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

## Q1: moda de 
**Resposta:** 

In [ ]:
print(df.industry.mode()[0])

## Q2: matriz de correlação
**Resposta:**  e  (0.916)

In [ ]:
corr = df[numerical].corr()
print(corr)

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(a, b, round(corr.loc[a, b], 3))

## Split

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

df_train = df_train.drop(columns='converted').reset_index(drop=True)
df_val = df_val.drop(columns='converted').reset_index(drop=True)
df_test = df_test.drop(columns='converted').reset_index(drop=True)

## Q3: informação mútua (só treino)
**Resposta:**  (0.03)

In [ ]:
mi = {c: round(mutual_info_score(df_train[c], y_train), 2) for c in categorical}
print(pd.Series(mi).sort_values(ascending=False))

## Q4: regressão logística
**Resposta:** 0.65 (accuracy 0.645)

In [ ]:
def train_and_score(features, C=1.0, digits=None):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    acc = (model.predict(X_val) == y_val).mean()
    return acc if digits is None else round(acc, digits)

all_features = categorical + numerical

acc_full = train_and_score(all_features)
print('Q4:', acc_full)

## Q5: eliminação de features
**Resposta:**  (diferença 0.002)

In [ ]:
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    features = [c for c in all_features if c != f]
    acc = train_and_score(features)
    print(f, '| acc sem a feature:', round(acc, 4), '| diferença:', round(acc_full - acc, 4))

## Q6: regularização
**Resposta:** C = 0.001 (accuracy 0.645)

In [ ]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, train_and_score(all_features, C=C, digits=3))